### Eavesdropping Scenario
#### Intercept and resend attack
Here we add an eavesdropper (Eve) that will intercept resend the circuit between Alice and Bob.
We expect to have 

In [1]:
import numpy as np
from qiskit_aer.primitives import SamplerV2
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister


In [2]:
sampler=SamplerV2()

In [3]:
def generate_bits(n):
    return np.random.randint(2, size=n)

In [4]:
def generate_bases(n):
    return np.random.choice(['Z','X'], 
                             size=n)

In [5]:
def encode_qubits(bits, bases):
    qr=QuantumRegister(len(bits), "q")
    cr=ClassicalRegister(len(bits), "c")
    qc=QuantumCircuit(qr,cr)
    for i, (bit,basis) in enumerate(zip(bits,bases)):
        if bit==1:
            qc.x(i)
        if basis=='X': 
        #apply hadamards gate
            qc.h(i)
    return qc


In [6]:
def eve_intercept_measure(qc, eve_bases, n):
    qr=qc.qregs[0]
    cr=ClassicalRegister(n, "c_eve")
    qc_eve=QuantumCircuit(qr,cr)
    qc_eve.compose(qc,inplace=True)
    for i in range(n):
        if eve_bases[i]=='X':
            qc_eve.h(i)
        qc_eve.measure(i,i)
    job=sampler.run([qc_eve], shots=1)
    res=job.result()[0].data.c_eve.get_int_counts()
    measured=max(res, key=res.get)
    bitstring=format(measured, f'0{n}b')
    eve_bits=np.array(list(map(int, bitstring[::-1])))
    return eve_bits

In [7]:
def eve_resend(eve_bits,eve_bases):
    n=len(eve_bits)
    qr=QuantumRegister(n, "q")
    cr=ClassicalRegister(n, "c")
    qc=QuantumCircuit(qr,cr)
    for i in range(n):
        if eve_bits[i]==1:
            qc.x(i)
        if eve_bases[i]=='X':
            qc.h(i)
    return qc

In [8]:
def measure_qubits(qc,bases):
    n=len(bases)
    qr=qc.qregs[0]
    cr=ClassicalRegister(n,"c_bob")
    mqc=QuantumCircuit(qr,cr)
    mqc.compose(qc, inplace=True)

    for i in range(n):
        if bases[i]=='X':
           mqc.h(i)
        mqc.measure(i,i)

    job=sampler.run([mqc], shots=1)
    res=job.result()[0].data.c_bob.get_int_counts()
    measured=max(res, key=res.get)
    bitstring=format(measured, f'0{n}b')
    return np.array(list(map(int,bitstring[::-1])))

In [9]:
def sift_keys(alice_bits, alice_bases, bob_bits, bob_bases):
    mask=alice_bases==bob_bases
    return alice_bits[mask], bob_bits[mask]

In [10]:
def qber_calc(alice_key, bob_key):
    if len(alice_key)==0:
        return 0
    return np.sum(alice_key!=bob_key)/len(alice_key)

In [11]:
def key_agreement_rate(alice_key, bob_key):
    return np.sum(alice_key==bob_key)/len(alice_key)

In [12]:
def key_generation_rate(sifted_len, total_qubs):
    return sifted_len/total_qubs

In [13]:
def run_bb84_eve(n=1000):
    alice_bits=generate_bits(n)
    alice_bases=generate_bases(n)
    qc_alice=encode_qubits(alice_bits, alice_bases)
    
    eve_bases=generate_bases(n)
    eve_bits=eve_intercept_measure(qc_alice, eve_bases, n)
    qc_after_eve=eve_resend(eve_bits, eve_bases)

    bob_bases=generate_bases(n)
    bob_bits=measure_qubits(qc_after_eve, bob_bases)

    alice_key, bob_key=sift_keys(
        alice_bits, alice_bases, bob_bits, bob_bases
    )
    qber=qber_calc(alice_key, bob_key)
    agreement=key_agreement_rate(alice_key, bob_key)
    kgr=key_generation_rate(len(alice_key), n)

    return {
        "QBER": qber,
        "agreement": agreement,
        "keyRate": kgr,
        "keyLength": len(alice_key),
    }

In [14]:
def run_exp_eve(runs=100, n=1000):
    qbers = []
    agreements = []
    key_rates = []

    for _ in range(runs):
        res = run_bb84_eve(n)
        qbers.append(res["QBER"])
        agreements.append(res["agreement"])
        key_rates.append(res["keyRate"])

    return {
        "QBER_mean": np.mean(qbers),
        "QBER_std": np.std(qbers),
        "agreement_mean": np.mean(agreements),
        "keyRate_mean": np.mean(key_rates),
    }

In [15]:
res = run_exp_eve(100, 1000)

In [16]:
print("=== Eavesdropping Channel Results ===")
for k, v in res.items():
    print(f"{k}: {v:.4f}")

=== Eavesdropping Channel Results ===
QBER_mean: 0.2516
QBER_std: 0.0196
agreement_mean: 0.7484
keyRate_mean: 0.4995
